## 3.2 Validation Sample Political Videos

**Author:** Kristina Aleksandrovna Pedersen

This file shows how validation samples of political videos we're generated and distributed across RAs. Note that we sought to balance the amount of videos across each party for each RA file.

In [1]:
#Loading relevant modules
import pandas as pd
import re
import os
import shutil
from glob import glob
#from functions import *
import random

In [2]:
df = pd.read_pickle("TikTok_accounts.pkl")
print(df.shape)
df.head()

(391, 6)


,type,name,party,url,user,party_cat
0,Parteien,CSU,CSU,https://www.tiktok.com/@csuauftiktok,@csuauftiktok,CDU_CSU
1,Parteien,SPD,SPD,https://www.tiktok.com/@deinespd,@deinespd,SPD
2,Parteien,CDU,CDU,https://www.tiktok.com/@insidecdu,@insidecdu,CDU_CSU
3,Parteien,Bündnis 90/Die Grünen,Grüne,https://www.tiktok.com/@diegruenen,@diegruenen,Grüne
4,Parteien,Die LINKE,Linke,https://www.tiktok.com/@die.linke,@die.linke,Linke


In [3]:
#Party categories
df.party_cat.unique()

array(['CDU_CSU', 'SPD', 'Grüne', 'Linke', 'BSW', 'FDP', 'AfD'],
      dtype=object)

### Validtion sample 1
*The first validation sample was created in preparation for the pilot test. This sample is also based on a smaller magnitude of content collected (spanning to January 2025).*

In [ ]:
#Number of RAs to distribute samples between
num_ras = 6
#Folder of party folders containing video metadata
party_folders = glob("./video_metadata_clean/*")

#Minimum numbre of observations per party
min_size = 14

In [6]:
#Empty list
dfs = []

#Iterating through each party folder
for folder in party_folders:
    #Getting all user files in folder
    user_files = glob(f"{folder}/*")
    #party category string
    party_cat = re.search("./video_metadata_clean/(.*)", folder).group(1)

    #Loading all user level files for party into one dataframe
    user_data = pd.concat([pd.read_pickle(file) for file in user_files], ignore_index=True)

    #Get number of observertions for party
    data_size = user_data.shape[0]
    #If the number of party observations is sufficient for each RA to get
    #the minimum size sample (=14) 
    if data_size >= (min_size*6):
        #then sample size is equal to minimum size of 14
        sample_size = min_size
    else:
        #else divide the number of party observations by the number of RAs
        sample_size = round(data_size/(num_ras+1))

    #for each RA
    for ra in range(1, num_ras+1):
        #create a sample equal to the sample size defined above
        sample = user_data.sample(sample_size, random_state = 123)
        #drop these observations from the dataframe
        user_data.drop(index = sample.index, inplace = True)
        
        #Create RA ID number column
        sample['ra_num'] = ra
        #Party category column
        sample['party_cat'] = party_cat
        #append dataframe to list
        dfs.append(sample)

#Concatente all the samples
sampledf = pd.concat(dfs, ignore_index = True)
sampledf = sampledf[['ra_num', 'party_cat', 'user_name', 'author_id', 'video_id' , 'video_date', 'url']]
sampledf[['anger', 'fear', 'superiority']] = None, None, None
print(sampledf.shape)
sampledf.head(3)

(510, 10)


,ra_num,party_cat,user_name,author_id,video_id,video_date,url,anger,fear,superiority
0,1,Linke,gregorgysi48,7212980941420823558,7436429584435825952,2024-11-12,https://www.tiktok.com/@gregorgysi48/video/743...,None,None,None
1,1,Linke,die.linke,7263426249372238881,7436797984504859936,2024-11-13,https://www.tiktok.com/@die.linke/video/743679...,None,None,None
2,1,Linke,dietmarbartsch,7171725835047683078,7443005970059595010,2024-11-30,https://www.tiktok.com/@dietmarbartsch/video/7...,None,None,None


In [ ]:
#Save each RA sample in a separate file
for ra in sampledf["ra_num"].unique():
    subdf = sampledf.query(f"ra_num == {ra}")
    subdf.to_excel(f"./samples/ra_{ra}_political.xlsx", index = False)

*Note: Numbers 1-6 were randomly assigned to each coder (file 3.1). The samples were further copied into a separate folder to get two coders of each video. Here the coders were assigned a new number (+1) and assigned the corresponding sample file.*


### Validation sample 2

In [6]:
#list of videos already manually coded in round 1
coded_videos = list(set(pd.concat([pd.read_excel(file) for file in glob("../samples/*") if "political" in file]).video_id.tolist()))
len(coded_videos)

510

In [ ]:
#Number of sample files
num_samples = 12
#Sample size per party
sample_size = 15
#Number of parties
num_parties = 7

In [8]:
#Empty list
dfs = []

#Iterating through each party folder
for folder in party_folders:
    #All user level files
    user_files = glob(f"{folder}/*")
    #party category string
    party_cat = re.search("../video_metadata_clean/(.*)", folder).group(1)

    #load the user level data
    user_data = pd.concat([pd.read_pickle(file) for file in user_files])
    # remove videos already coded by RAs
    user_data = user_data[~user_data.video_id.isin(coded_videos)].reset_index(drop = True)

    #for each sample file
    for file in range(1, num_samples+1):
        try:
            #take a random sample of party videos
            sample = user_data.sample(sample_size, random_state = 123)
        except:
            #Error will be thrown if there aren't enough observations
            #In such a case divide the number of observations with the iteration number
            #(for better distribution should have been the remaining number of iterations = num_samples - file)
            sample_size_alt = round(user_data.shape[0]/(file +1))
            sample = user_data.sample(sample_size_alt, random_state = 123)

        #remove the sampled observations from user dataframe
        user_data.drop(index = sample.index, inplace = True)

        #File number column
        sample['file_num'] = file
        #Party column
        sample['party_cat'] = party_cat
        #Append to list
        dfs.append(sample)

#Concatenate
sampledf = pd.concat(dfs, ignore_index = True)
sampledf = sampledf[['file_num', 'party_cat', 'user_name', 'author_id', 'video_id' , 'video_date', 'url']]
sampledf[['anger', 'fear', 'superiority', 'clear_party', 'notes']] = [None, None,None, None, None]
print(sampledf.shape)
sampledf.head(3)

(1049, 12)


,file_num,party_cat,user_name,author_id,video_id,video_date,url,anger,fear,superiority,clear_party,notes
0,1,Linke,pellmann,6575513889865007109,7443125900603903254,2024-11-30,https://www.tiktok.com/@pellmann/video/7443125...,None,None,None,None,None
1,1,Linke,die.linke,7263426249372238881,7463436593290857750,2025-01-24,https://www.tiktok.com/@die.linke/video/746343...,None,None,None,None,None
2,1,Linke,die_linke_brandenburg,7400354806085551136,7463420200151485729,2025-01-24,https://www.tiktok.com/@die_linke_brandenburg/...,None,None,None,None,None


In [10]:
#Each sample seperated into each own file and saved to .xlsx format for manual coding
for file in sampledf["file_num"].unique():
    subdf = sampledf.query(f"file_num == {file}")
    print(file, subdf.shape)
    subdf.to_excel(f"../samples2/file_{file}.xlsx", index = False)

1 (105, 12)
2 (95, 12)
3 (92, 12)
4 (91, 12)
5 (91, 12)
6 (91, 12)
7 (90, 12)
8 (90, 12)
9 (76, 12)
10 (76, 12)
11 (76, 12)
12 (76, 12)


*The coders were allowed to chose sample files based on sample sizes and their availibility this time around. Similarily, to the first round of coding described above the samples were copied into a seperate folder for validation, so each sample was coded by at least two seperate RAs. The section below shows the (anonymized) distribution of coders across all sample filesm including the third round of validation coding initiated shortly before the survey launch.*

### Coder distribution across samples and coding rounds

In [82]:
coders = pd.read_pickle("../samples_validation_final/coder_sheet_anonymous.pkl")
coders

,Sample file,Sample size,CoderID R1,CoderID R2,CoderID R3
0,ra_1_political.xlsx,85,1,6,2
1,ra_2_political.xlsx,85,2,1,5
2,ra_3_political.xlsx,85,3,2,5
3,ra_5_political.xlsx,85,5,4,2
4,ra_4_political.xlsx,85,4,3,5
5,ra_6_political.xlsx,85,6,8,1
6,file_11.xlsx,76,7,9,8
7,file_5.xlsx,91,1,4,6
8,file_1.xlsx,105,4,3,1
9,file_9.xlsx,76,1,9,4


In [83]:
#Complete number of observations coded
coders['Sample size'].sum()

np.int64(1559)